# **CLIP AI Image Classifier**
Train on 5 generators (trainval split), evaluate per-dataset on test split.
Architecture: CLIPVisionModel (HuggingFace) + Linear head, last 2 layers unfrozen.
Loss: BCEWithLogitsLoss + mixed precision (AMP).

In [ ]:
# Block 0 - Mount and install
from google.colab import drive
drive.mount('/content/drive')
!pip install -q transformers

Mounted at /content/drive


In [ ]:
# Block 1 - Copy training images from Drive to VM for fast local I/O
import os
import shutil
import pandas as pd
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

INVENTORY_PATH = "/content/drive/MyDrive/TrainingData/dataset_inventory.csv"
LOCAL_TRAIN_DIR = "/content/local_train_images"
LOCAL_TRAIN_CSV = "/content/local_train.csv"
SAVE_DIR = "/content/drive/MyDrive/Model/clip_classification_v5"

DATASETS = [
    'imagenet_ai_0424_sdv5',
    'imagenet_ai_0419_vqdm',
    'imagenet_ai_0508_adm',
    'imagenet_ai_0419_biggan',
    'imagenet_glide',
]

os.makedirs(LOCAL_TRAIN_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

df_full = pd.read_csv(INVENTORY_PATH)

# Filter only the 5 target datasets, trainval split
df_trainval = df_full[
    (df_full['dataset'].isin(DATASETS)) &
    (df_full['split'] == 'trainval')
].copy().reset_index(drop=True)

print(f"Total trainval samples: {len(df_trainval):,}")
print(df_trainval.groupby(['dataset', 'is_fake']).size().to_string())

# Copy files to VM using 16 threads for maximum speed
def copy_single(args):
    idx, row = args
    dest = os.path.join(LOCAL_TRAIN_DIR, f"{idx}_{row['file_name']}")
    if not os.path.exists(dest):
        try:
            shutil.copy2(row['file_path'], dest)
        except Exception:
            return None
    return dest

print("Copying trainval images to VM...")
with ThreadPoolExecutor(max_workers=16) as executor:
    new_paths = list(tqdm(
        executor.map(copy_single, df_trainval.iterrows()),
        total=len(df_trainval)
    ))

df_trainval['local_path'] = new_paths
df_trainval = df_trainval.dropna(subset=['local_path'])
df_trainval.to_csv(LOCAL_TRAIN_CSV, index=False)
print(f"Done. {len(df_trainval):,} images ready on VM.")

Total trainval samples: 10,000
dataset                  is_fake
imagenet_ai_0419_biggan  0          1000
                         1          1000
imagenet_ai_0419_vqdm    0          1000
                         1          1000
imagenet_ai_0424_sdv5    0          1000
                         1          1000
imagenet_ai_0508_adm     0          1000
                         1          1000
imagenet_glide           0          1000
                         1          1000
Copying trainval images to VM...


  0%|          | 0/10000 [00:00<?, ?it/s]

Done. 10,000 images ready on VM.


In [ ]:
# Block 2 - Imports and config
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from collections import defaultdict
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from transformers import CLIPVisionModel

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_PATH = "/content/drive/MyDrive/Model/clip_model"

BATCH_SIZE  = 128   # Larger batch with AMP for speed
EPOCHS      = 5
LR          = 2e-5
VAL_SPLIT   = 0.15
SEED        = 42

torch.manual_seed(SEED)
print(f"Device: {DEVICE}")

# CLIP preprocessing - matches HuggingFace CLIPVisionModel expected input
clip_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.48145466, 0.4578275, 0.40821073],
        std=[0.26862954, 0.26130258, 0.27577711]
    )
])

Device: cuda


In [ ]:
# Block 3 - Model architecture (same as v2)
class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super().__init__()
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)
        hidden_size = self.vision_encoder.config.hidden_size
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        return self.classifier(outputs.pooler_output)


model = ClassificationCLIP(MODEL_PATH).to(DEVICE)

# Freeze all backbone params first
for param in model.vision_encoder.parameters():
    param.requires_grad = False

# Unfreeze last 2 transformer layers (same as v2, ~25M trainable)
for layer in model.vision_encoder.vision_model.encoder.layers[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Always unfreeze the classification head
for param in model.classifier.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} params")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
visual_projection.weight                                     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.o

Trainable: 25,193,473 / 303,180,801 params


In [ ]:
# Block 4 - Dataset and DataLoaders
class ImageDataset(Dataset):
    # is_fake=1 -> fake (label=0 for real_image convention)
    # is_fake=0 -> real (label=1 for real_image convention)
    def __init__(self, df, transform, path_col='local_path'):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.path_col  = path_col

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # real_image: 1=real, 0=fake (matches BCEWithLogitsLoss convention)
        label      = float(1 - row['is_fake'])  # is_fake=1->label=0, is_fake=0->label=1
        dataset    = row['dataset']
        try:
            img = Image.open(row[self.path_col]).convert('RGB')
            img = self.transform(img)
        except Exception:
            img = torch.zeros(3, 224, 224)
        return img, torch.tensor([label], dtype=torch.float32), dataset


df_train_csv = pd.read_csv(LOCAL_TRAIN_CSV)
full_ds      = ImageDataset(df_train_csv, clip_transform)

val_n   = int(len(full_ds) * VAL_SPLIT)
train_n = len(full_ds) - val_n
train_ds, val_ds = random_split(
    full_ds, [train_n, val_n],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

print(f"Train: {train_n:,} | Val: {val_n:,}")

Train: 8,500 | Val: 1,500


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# Block 5 - Training with AMP (mixed precision for max speed)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=1e-2
)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)
scaler = torch.amp.GradScaler('cuda')


@torch.no_grad()
def validate(model, loader):
    model.eval()
    correct = total = 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
        preds   = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return correct / total


best_val_acc = 0.0
print(f"Training on {len(train_ds):,} samples, validating on {len(val_ds):,}")
print("=" * 60)

for epoch in range(EPOCHS):
    model.train()
    run_loss = correct = total = 0

    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for imgs, labels, _ in bar:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        run_loss += loss.item()
        preds     = (torch.sigmoid(logits.detach()) >= 0.5).float()
        correct  += (preds == labels).sum().item()
        total    += labels.size(0)
        bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.2%}")

    val_acc = validate(model, val_loader)
    scheduler.step(val_acc)

    print(f"Epoch {epoch+1}  loss={run_loss/len(train_loader):.4f}  "
          f"train={correct/total:.2%}  val={val_acc:.2%}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(),
                   os.path.join(SAVE_DIR, 'best_model.pth'))
        print(f"  Best saved: val={best_val_acc:.2%}")

print(f"\nTraining done. Best val: {best_val_acc:.2%}")

Training on 8,500 samples, validating on 1,500


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 1/5:   0%|          | 0/67 [00:00<?, ?it/s]

Epoch 1  loss=0.3381  train=84.49%  val=94.60%
  Best saved: val=94.60%


Epoch 2/5:   0%|          | 0/67 [00:00<?, ?it/s]

...

Epoch 3  loss=0.0176  train=99.51%  val=97.87%
  Best saved: val=97.87%


Epoch 4/5:   0%|          | 0/67 [00:00<?, ?it/s]

Epoch 4  loss=0.0052  train=99.88%  val=97.53%


Epoch 5/5:   0%|          | 0/67 [00:00<?, ?it/s]

Epoch 5  loss=0.0056  train=99.80%  val=98.13%
  Best saved: val=98.13%

Training done. Best val: 98.13%


In [ ]:
# Block 6 - Per-dataset evaluation on test split
# Load best checkpoint
model.load_state_dict(torch.load(
    os.path.join(SAVE_DIR, 'best_model.pth'),
    map_location=DEVICE, weights_only=False
))
model.eval()

df_full = pd.read_csv(INVENTORY_PATH)
df_test = df_full[
    (df_full['dataset'].isin(DATASETS)) &
    (df_full['split'] == 'test')
].copy().reset_index(drop=True)

print(f"Test samples: {len(df_test):,}")

class TestDataset(Dataset):
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = float(1 - row['is_fake'])
        try:
            img = Image.open(row['file_path']).convert('RGB')
            img = self.transform(img)
        except Exception:
            img = torch.zeros(3, 224, 224)
        return img, torch.tensor([label], dtype=torch.float32), row['dataset'], int(row['is_fake'])


test_ds = TestDataset(df_test, clip_transform)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                     num_workers=4, pin_memory=True)

# Collect predictions
results = defaultdict(lambda: {'correct': 0, 'total': 0,
                                'real_correct': 0, 'real_total': 0,
                                'fake_correct': 0, 'fake_total': 0})

with torch.no_grad():
    for imgs, labels, datasets, is_fakes in tqdm(test_dl, desc="Evaluating"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
        preds   = (torch.sigmoid(logits) >= 0.5).float()
        correct = (preds == labels)

        for i in range(len(labels)):
            ds      = datasets[i]
            is_fake = is_fakes[i].item()
            ok      = correct[i].item()

            results[ds]['total']   += 1
            results[ds]['correct'] += ok

            if is_fake:
                results[ds]['fake_total']   += 1
                results[ds]['fake_correct'] += ok
            else:
                results[ds]['real_total']   += 1
                results[ds]['real_correct'] += ok

# Print per-dataset report
print("\n" + "=" * 65)
print(f"{'Dataset':<30} {'Overall':>8} {'Real':>8} {'Fake':>8}")
print("-" * 65)
total_correct = total_all = 0
for ds in DATASETS:
    r = results[ds]
    overall = r['correct'] / r['total'] * 100 if r['total'] else 0
    real    = r['real_correct'] / r['real_total'] * 100 if r['real_total'] else 0
    fake    = r['fake_correct'] / r['fake_total'] * 100 if r['fake_total'] else 0
    total_correct += r['correct']
    total_all     += r['total']
    print(f"{ds:<30} {overall:>7.2f}%  {real:>7.2f}%  {fake:>7.2f}%")
print("-" * 65)
print(f"{'TOTAL':<30} {total_correct/total_all*100:>7.2f}%")
print("=" * 65)

Test samples: 2,000


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Evaluating:   0%|          | 0/16 [00:00<?, ?it/s]


Dataset                         Overall     Real     Fake
-----------------------------------------------------------------
imagenet_ai_0424_sdv5            95.50%    97.00%    94.00%
imagenet_ai_0419_vqdm            97.25%    98.00%    96.50%
imagenet_ai_0508_adm             99.25%    99.00%    99.50%
imagenet_ai_0419_biggan          99.25%    98.50%   100.00%
imagenet_glide                   98.50%    97.00%   100.00%
-----------------------------------------------------------------
TOTAL                            97.95%


In [ ]:
# Add two new columns to inventory for this model's predictions

all_prob_fake = []
with torch.no_grad():
    for imgs, labels, datasets, is_fakes in tqdm(test_dl, desc="Collecting predictions"):
        imgs = imgs.to(DEVICE)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(imgs)
        prob_fake = (1 - torch.sigmoid(logits)).squeeze(1).cpu().float().numpy()
        all_prob_fake.extend(prob_fake.tolist())

df_test['full_data_pred_label']     = [1 if p >= 0.5 else 0 for p in all_prob_fake]
df_test['full_data_pred_confident'] = [round(p, 4) for p in all_prob_fake]

# Read inventory, add columns if not exist, update test rows only
df_full = pd.read_csv(INVENTORY_PATH)

if 'full_data_pred_label' not in df_full.columns:
    df_full['full_data_pred_label'] = None

if 'full_data_pred_confident' not in df_full.columns:
    df_full['full_data_pred_confident'] = None

# Update only the test rows by matching file_path
df_full = df_full.set_index('file_path')
df_test_indexed = df_test.set_index('file_path')

df_full.loc[df_test_indexed.index, 'full_data_pred_label']     = df_test_indexed['full_data_pred_label']
df_full.loc[df_test_indexed.index, 'full_data_pred_confident'] = df_test_indexed['full_data_pred_confident']

df_full = df_full.reset_index()
df_full.to_csv(INVENTORY_PATH, index=False)

print(f"Saved to: {INVENTORY_PATH}")
print(f"Rows updated: {df_full['full_data_pred_label'].notna().sum():,}")

Saved to: /content/drive/MyDrive/TrainingData/dataset_inventory.csv
Rows updated: 2,000
